# The SOLAR-A Mission: An Overview — Implementation / 구현

**Paper**: Ogawara, Y. et al. (1991). "The SOLAR-A Mission: An Overview", *Solar Physics* **136**, 1–16. DOI: 10.1007/BF00151692

This notebook implements quantitative analyses of SOLAR-A (YOHKOH) mission design:

1. **Instrument suite specifications** (HXT/SXT/BCS/WBS) — visualize energy/wavelength coverage
2. **Orbit visualization** — 600 km / 31° / 97 min near-circular LEO
3. **Telemetry & data-recorder budget** — overwrite priority dynamics
4. **Bragg crystal spectrometer Doppler resolution** — applied to evaporation upflows
5. **Fourier-synthesis (HXT) imaging concept** — toy reconstruction from $(u,v)$ samples
6. **Automated flare-mode state machine** — triple-threshold logic simulation
7. **Solar cycle 22 GOES X-ray flux context** — synthetic illustration

이 노트북은 SOLAR-A(YOHKOH)의 임무 설계 핵심 정량 요소를 구현/시각화합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## Part 1: Instrument Suite Energy/Wavelength Coverage / 기기 에너지·파장 커버리지

Visualize the spectral coverage of HXT, SXT, BCS, and WBS as overlapping bands on a log-energy axis. The four instruments together span roughly 0.2 keV (3–60 Å soft X-rays) to 100 MeV (BGO gamma-ray) — five and a half orders of magnitude.

HXT, SXT, BCS, WBS의 스펙트럼 범위를 로그 에너지 축에 겹쳐 그려 SOLAR-A 전체 커버리지가 ~0.2 keV에서 100 MeV(약 5.5 자릿수)에 걸친다는 점을 보입니다.

In [ ]:
def angstrom_to_keV(lambda_A: float) -> float:
    """Convert wavelength in angstroms to photon energy in keV.

    Args:
        lambda_A: Wavelength in angstroms.

    Returns:
        Photon energy in keV (E = hc/lambda, with hc = 12.398 keV*A).
    """
    return 12.398 / lambda_A


# SXT covers 3-60 angstrom -> ~0.207 to 4.13 keV
sxt_keV = (angstrom_to_keV(60.0), angstrom_to_keV(3.0))
print(f"SXT: 3-60 A  ->  {sxt_keV[0]:.3f} - {sxt_keV[1]:.3f} keV")

# BCS lines (S XV, Ca XIX, Fe XXV, Fe XXVI)
bcs_lines = {
    'S XV':    5.0385,
    'Ca XIX':  3.1769,
    'Fe XXV':  1.8509,
    'Fe XXVI': 1.7780,
}
for name, lam in bcs_lines.items():
    print(f"BCS {name:7s}: lambda={lam} A  ->  E={angstrom_to_keV(lam):.3f} keV")

In [ ]:
# Build instrument bands on a log-energy axis.
instruments = [
    # (name, low_keV, high_keV, color, y_position)
    ('SXT (3-60 A)',          sxt_keV[0],  sxt_keV[1],  'tab:blue',    4),
    ('BCS (Fe/Ca/S lines)',   2.5,         8.0,         'tab:cyan',    3),
    ('WBS-SXS (gas)',         2.0,         30.0,        'tab:green',   2),
    ('HXT (4 bands)',         15.0,        100.0,       'tab:orange',  1),
    ('WBS-HXS (NaI)',         20.0,        400.0,       'tab:red',     0),
    ('WBS-GRS (BGO)',         200.0,       1.0e5,       'tab:purple', -1),
]

fig, ax = plt.subplots(figsize=(11, 5))
for name, lo, hi, color, y in instruments:
    ax.barh(y, hi - lo, left=lo, height=0.7, color=color, alpha=0.75,
            edgecolor='k')
    ax.text(np.sqrt(lo * hi), y, name, ha='center', va='center', fontsize=10)

# Mark HXT energy boundaries 15-24-35-57-100 keV.
for e in [15, 24, 35, 57, 100]:
    ax.axvline(e, ls=':', color='gray', alpha=0.4)

ax.set_xscale('log')
ax.set_xlim(0.1, 2e5)
ax.set_ylim(-2, 5)
ax.set_yticks([])
ax.set_xlabel('Photon Energy (keV)')
ax.set_title('SOLAR-A Spectral Coverage — Five Decades (HXT/SXT/BCS/WBS)\n'
             'SOLAR-A 스펙트럼 커버리지 — 5자릿수')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: Orbit Visualization / 궤도 시각화

SOLAR-A's orbit is near-circular at altitude $h = 600$ km, inclination $i = 31^\circ$, period $T \approx 97$ min. Using Kepler's third law:

$$T = 2\pi\sqrt{\frac{(R_\oplus + h)^3}{GM_\oplus}}$$

We verify the period and visualize one orbit in 3D.

Kepler 제3법칙으로 600 km 고도 LEO 주기를 검증하고 한 궤도를 3D로 시각화합니다.

In [ ]:
def orbital_period(h_km: float, R_E_km: float = 6378.0,
                   GM_E: float = 3.986e14) -> float:
    """Compute circular-orbit period from altitude.

    Args:
        h_km: Altitude above Earth's surface in km.
        R_E_km: Earth radius in km.
        GM_E: Earth's gravitational parameter in m^3/s^2.

    Returns:
        Orbital period in seconds.
    """
    a_m = (R_E_km + h_km) * 1.0e3
    return 2.0 * np.pi * np.sqrt(a_m**3 / GM_E)


T_s = orbital_period(600.0)
print(f"Period at 600 km altitude: {T_s:.1f} s = {T_s/60.0:.2f} min")
print(f"Paper quotes: ~97 min  (delta = {T_s/60.0 - 97.0:+.2f} min)")

In [ ]:
def orbit_xyz(h_km: float = 600.0, inc_deg: float = 31.0,
              n_points: int = 360, R_E_km: float = 6378.0) -> np.ndarray:
    """Generate (x, y, z) coordinates for one circular orbit.

    Args:
        h_km: Altitude in km.
        inc_deg: Inclination in degrees.
        n_points: Number of samples per orbit.
        R_E_km: Earth radius in km.

    Returns:
        Array of shape (3, n_points) in km, with the rotation axis along z.
    """
    a = R_E_km + h_km
    nu = np.linspace(0.0, 2.0 * np.pi, n_points)
    inc = np.deg2rad(inc_deg)
    # Orbit lies in a plane tilted by inc about the x-axis.
    x = a * np.cos(nu)
    y = a * np.sin(nu) * np.cos(inc)
    z = a * np.sin(nu) * np.sin(inc)
    return np.vstack([x, y, z])


fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Earth as a sphere.
u, v = np.mgrid[0:2*np.pi:60j, 0:np.pi:30j]
R_E = 6378.0
xs = R_E * np.cos(u) * np.sin(v)
ys = R_E * np.sin(u) * np.sin(v)
zs = R_E * np.cos(v)
ax.plot_surface(xs, ys, zs, color='lightblue', alpha=0.4, edgecolor='none')

# SOLAR-A orbit.
orb = orbit_xyz(h_km=600.0, inc_deg=31.0)
ax.plot(orb[0], orb[1], orb[2], color='crimson', lw=2,
        label='SOLAR-A (600 km, 31 deg)')

# Equatorial plane reference.
phi = np.linspace(0, 2*np.pi, 200)
ax.plot(R_E*np.cos(phi), R_E*np.sin(phi), np.zeros_like(phi),
        color='black', ls='--', lw=0.7, alpha=0.6, label='Equator')

ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.set_title('SOLAR-A LEO Orbit / SOLAR-A 저궤도\n'
             'h = 600 km, i = 31 deg, T = 97 min')
ax.legend(loc='upper right')
ax.set_box_aspect([1, 1, 1])
plt.tight_layout()
plt.show()

## Part 3: Telemetry / Data Recorder Budget / 텔레메트리·기록기 예산

SOLAR-A's bubble data recorder (BDR) holds 10 Mbyte. At the high telemetry rate (32 kbps), it fills in:

$$\tau_{\text{fill}} = \frac{10 \times 8 \times 10^6 \text{ bits}}{32\,000 \text{ bits/s}} \approx 2500 \text{ s} \approx 41.7 \text{ min}$$

Since the daylit fraction of one ~97-min orbit is ~58 min, the recorder cannot store an entire daylit pass at high rate, motivating onboard overwrite-priority logic.

10 Mbyte BDR가 32 kbps에서 ~42분 만에 가득 차므로, 60분 이상 일조 시간 동안 우선순위 기반 overwrite 보호가 필수임을 수치로 확인합니다.

In [ ]:
def bdr_fill_time(capacity_Mbyte: float, rate_kbps: float) -> float:
    """Compute time to fill a recorder at a constant data rate.

    Args:
        capacity_Mbyte: Recorder capacity in Mbyte.
        rate_kbps: Constant ingest rate in kbps.

    Returns:
        Fill time in seconds.
    """
    capacity_bits = capacity_Mbyte * 8.0 * 1.0e6
    return capacity_bits / (rate_kbps * 1.0e3)


for rate in [32, 4, 1]:
    t_fill = bdr_fill_time(10.0, rate)
    print(f"Rate {rate:2d} kbps:  fill time = {t_fill/60.0:6.1f} min")

t_dump = bdr_fill_time(10.0, 262.0)
print(f"Dump @ 262 kbps:    {t_dump:.1f} s = {t_dump/60.0:.2f} min")

In [ ]:
# Visualize fill time vs daylit fraction across one orbit.
T_orbit_min = 97.0
daylit_frac = 0.6
daylit_min = T_orbit_min * daylit_frac

rates = np.array([1, 4, 32])  # kbps
fill_min = np.array([bdr_fill_time(10.0, r) / 60.0 for r in rates])

fig, ax = plt.subplots()
bars = ax.bar([f'{r} kbps' for r in rates], fill_min,
              color=['tab:green', 'tab:orange', 'tab:red'])
ax.axhline(daylit_min, ls='--', color='black',
           label=f'Daylit pass ~{daylit_min:.0f} min')
ax.set_ylabel('BDR fill time [min]')
ax.set_title('10 Mbyte Bubble Data Recorder — Fill Time vs Telemetry Rate\n'
             '10 MB BDR — 텔레메트리율별 기록 한계')
for bar, val in zip(bars, fill_min):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5,
            f'{val:.1f} min', ha='center')
ax.legend()
ax.set_ylim(0, max(fill_min) * 1.15)
plt.tight_layout()
plt.show()

## Part 4: Bragg Crystal Spectrometer Doppler Resolution / BCS 도플러 분해능

Bragg's law $n\lambda = 2d\sin\theta$ underlies BCS. Doppler velocity sensitivity:

$$\frac{\Delta v}{c} = \frac{\Delta\lambda}{\lambda}$$

Apply to the four BCS lines and check whether 300–400 km/s chromospheric evaporation upflows (Tanaka et al. 1983) are well-resolved.

BCS의 4개 라인에 대해 도플러 속도 분해능을 계산하고 chromospheric evaporation upflow(300–400 km/s)가 분해 가능한지 확인합니다.

In [ ]:
C_KMS = 2.998e5  # speed of light in km/s

# Table II values: (line, lambda_0_A, dlambda_mA, range_low_A, range_high_A)
bcs_table = [
    ('S XV',    5.0385, 3.232, 5.0160, 5.1143),
    ('Ca XIX',  3.1769, 0.918, 3.1631, 3.1912),
    ('Fe XXV',  1.8509, 0.710, 1.8298, 1.8942),
    ('Fe XXVI', 1.7780, 0.565, 1.7636, 1.8044),
]

print(f"{'Line':<8} {'lambda0[A]':>10} {'dlambda[mA]':>12} "
      f"{'R=l/dl':>8} {'dv[km/s]':>10} {'v_evap[km/s]':>14}")
print('-' * 70)
for line, lam, dlam_mA, _, _ in bcs_table:
    dlam = dlam_mA * 1e-3  # mA to A
    R = lam / dlam
    dv = C_KMS * dlam / lam
    # Range of resolvable velocities relative to line center:
    range_kmps = C_KMS * (bcs_table[0][4] - bcs_table[0][3]) / lam
    print(f"{line:<8} {lam:>10.4f} {dlam_mA:>12.3f} "
          f"{R:>8.0f} {dv:>10.1f}  ~3-400 km/s")

In [ ]:
# Visualize: each line's resolution element vs the typical evaporation upflow.
fig, ax = plt.subplots()
lines = [r[0] for r in bcs_table]
dv_kmps = [C_KMS * (r[2] * 1e-3) / r[1] for r in bcs_table]

ax.bar(lines, dv_kmps, color='tab:cyan', edgecolor='k', label='BCS dv (resolution)')
ax.axhline(300, ls='--', color='tab:red',
           label='Evaporation upflow ~300-400 km/s')
ax.axhline(400, ls='--', color='tab:red')
ax.fill_between([-0.5, 3.5], 300, 400, alpha=0.15, color='tab:red')
ax.set_xlim(-0.5, 3.5)
ax.set_ylabel('Velocity resolution [km/s]')
ax.set_title('BCS Doppler Resolution per Line vs Evaporation Upflow\n'
             'BCS 라인별 도플러 분해능 vs 증발 상승류')
ax.legend()
for x, val in enumerate(dv_kmps):
    ax.text(x, val + 5, f'{val:.0f}', ha='center')
plt.tight_layout()
plt.show()

print('\nAll four lines easily resolve the 300-400 km/s evaporation flow.')

## Part 5: Fourier-Synthesis Imaging Concept (HXT) / 푸리에 합성 영상 개념

HXT samples 32 complex Fourier components $V_i$ of the source brightness $I(x,y)$:

$$V_i = \iint I(x,y)\,e^{-2\pi j(u_i x + v_i y)}\,dx\,dy$$

We build a toy two-source flare image, sample 32 $(u, v)$ points, and demonstrate that even sparse Fourier sampling captures the source positions — illustrating the principle behind HXT's ~5″ imaging.

두 개의 footpoint를 가진 플레어 모형을 만들고 32개 $(u,v)$ 점에서 샘플링한 뒤 역변환하여, 소량의 푸리에 성분만으로도 source 위치가 드러남을 보입니다.

In [ ]:
def make_two_source_flare(N: int = 128, sep_arcsec: int = 30) -> np.ndarray:
    """Construct a toy two-footpoint flare image.

    Args:
        N: Image side length in pixels (1 pix = 1 arcsec).
        sep_arcsec: Angular separation between the two Gaussian sources.

    Returns:
        N x N image array.
    """
    x = np.arange(N) - N // 2
    X, Y = np.meshgrid(x, x)
    sigma = 3.0
    src1 = np.exp(-((X + sep_arcsec / 2)**2 + Y**2) / (2 * sigma**2))
    src2 = 0.7 * np.exp(-((X - sep_arcsec / 2)**2 + (Y - 5)**2) / (2 * sigma**2))
    return src1 + src2


def hxt_uv_samples(n_pairs: int = 32, kmin: float = 0.005,
                   kmax: float = 0.1) -> np.ndarray:
    """Build a synthetic HXT-like (u, v) sampling pattern.

    Args:
        n_pairs: Number of bigrid pairs (32 for HXT).
        kmin: Minimum spatial frequency (cycles/arcsec).
        kmax: Maximum spatial frequency.

    Returns:
        Array of shape (n_pairs, 2) containing (u, v) samples.
    """
    # Logarithmic radial spacing, evenly distributed angles.
    radii = np.logspace(np.log10(kmin), np.log10(kmax), n_pairs)
    angles = np.linspace(0, np.pi, n_pairs, endpoint=False)
    u = radii * np.cos(angles)
    v = radii * np.sin(angles)
    return np.column_stack([u, v])


def fourier_sample(image: np.ndarray, uv: np.ndarray) -> np.ndarray:
    """Sample Fourier-domain points from an image via DFT.

    Args:
        image: Input N x N image (1 pix = 1 arcsec).
        uv: Sampling positions in cycles/arcsec, shape (n, 2).

    Returns:
        Complex visibilities at the requested (u, v) points.
    """
    N = image.shape[0]
    x = np.arange(N) - N // 2
    X, Y = np.meshgrid(x, x)
    visibilities = np.zeros(len(uv), dtype=complex)
    for i, (u, v) in enumerate(uv):
        kernel = np.exp(-2j * np.pi * (u * X + v * Y))
        visibilities[i] = np.sum(image * kernel)
    return visibilities


def back_project(uv: np.ndarray, vis: np.ndarray, N: int = 128) -> np.ndarray:
    """Reconstruct an image by direct inverse summation of sampled visibilities.

    Note: This is a 'dirty image' equivalent — adequate to demonstrate the
    Fourier-synthesis principle but lacks deconvolution / CLEAN.

    Args:
        uv: Sampling positions (n, 2) in cycles/arcsec.
        vis: Complex visibilities at those points.
        N: Output image size.

    Returns:
        Real-valued reconstructed image (N, N).
    """
    x = np.arange(N) - N // 2
    X, Y = np.meshgrid(x, x)
    img = np.zeros((N, N), dtype=complex)
    for (u, v), V in zip(uv, vis):
        # Both phases (V and conj(V) at -u,-v) so result is real.
        img += V * np.exp(2j * np.pi * (u * X + v * Y))
        img += np.conj(V) * np.exp(2j * np.pi * (-u * X + -v * Y))
    return img.real / len(uv)

In [ ]:
true_image = make_two_source_flare()
uv = hxt_uv_samples(n_pairs=32)
vis = fourier_sample(true_image, uv)
dirty = back_project(uv, vis, N=true_image.shape[0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
im0 = axes[0].imshow(true_image, extent=[-64, 64, -64, 64],
                     origin='lower', cmap='inferno')
axes[0].set_title('True flare (two sources)\n진짜 영상')
axes[0].set_xlabel('x [arcsec]'); axes[0].set_ylabel('y [arcsec]')
fig.colorbar(im0, ax=axes[0], fraction=0.04)

axes[1].scatter(uv[:, 0], uv[:, 1], s=30, c='tab:red')
axes[1].scatter(-uv[:, 0], -uv[:, 1], s=30, c='tab:red')  # Hermitian pair
axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)
axes[1].set_xlabel('u [cycles/arcsec]'); axes[1].set_ylabel('v [cycles/arcsec]')
axes[1].set_title('HXT (u,v) sampling — 32 pairs\nHXT 푸리에 표본')
axes[1].set_aspect('equal')

im2 = axes[2].imshow(dirty, extent=[-64, 64, -64, 64],
                     origin='lower', cmap='inferno')
axes[2].set_title('Reconstruction (dirty image)\n복원 영상')
axes[2].set_xlabel('x [arcsec]'); axes[2].set_ylabel('y [arcsec]')
fig.colorbar(im2, ax=axes[2], fraction=0.04)
plt.tight_layout()
plt.show()

print('Two source positions are recovered despite only 32 visibility samples.')

## Part 6: Automated Flare-Mode State Machine / 자동 플레어 모드 상태 기계

SOLAR-A's DP uses three thresholds — **flare**, **flare-end**, **great-flare** — applied to flare-sensor counting rates (HXS, SXS, or BCS), plus an RBM cross-check, to control the observing mode (Fig. 2 & 3 of paper).

Logic:
1. quiet → flare-high when sensor crosses flare threshold AND RBM does not.
2. great-flare flag set when count exceeds great-flare threshold (data priority elevated).
3. flare-high → flare-medium when count drops below great-flare threshold but stays above flare-end (case B).
4. flare → quiet when count drops below flare-end threshold (case A).

We simulate a synthetic flare light curve and animate the mode transitions.

DP의 자동 모드 전환 로직(3중 임계값 + RBM 교차 확인)을 합성 광도곡선에 시뮬레이션합니다.

In [ ]:
def synthetic_flare_lightcurve(t: np.ndarray) -> np.ndarray:
    """Generate a synthetic two-flare HXS-like light curve.

    Args:
        t: Time array in seconds.

    Returns:
        Counting rate (counts/s) including a small flare and a great flare.
    """
    quiet = 50.0 + 5.0 * np.random.randn(len(t))
    # Small flare around t=400, peak ~600 cps.
    small = 600.0 * np.exp(-((t - 400) / 60.0)**2)
    # Great flare around t=1500, peak ~5000 cps with slow decay.
    rise = (t > 1400) & (t < 1500)
    decay = (t >= 1500)
    great = np.zeros_like(t)
    great[rise] = 5000.0 * ((t[rise] - 1400) / 100.0)**2
    great[decay] = 5000.0 * np.exp(-(t[decay] - 1500) / 400.0)
    return np.clip(quiet + small + great, 0, None)


def classify_modes(rate: np.ndarray,
                   flare_thresh: float = 250.0,
                   flare_end_thresh: float = 150.0,
                   great_flare_thresh: float = 2000.0) -> np.ndarray:
    """Run the SOLAR-A DP flare-mode state machine over a counting-rate curve.

    Args:
        rate: Counting rate vs time (1 sample = 1 s nominal).
        flare_thresh: Counting rate at which flare flag is set.
        flare_end_thresh: Counting rate at which flare flag is reset.
        great_flare_thresh: Counting rate at which 'great flare' priority is asserted.

    Returns:
        Integer mode array: 0 = quiet, 1 = flare-medium, 2 = flare-high.
    """
    mode = np.zeros_like(rate, dtype=int)
    flag_set = False
    great_flag = False
    for i, r in enumerate(rate):
        if not flag_set:
            if r > flare_thresh:
                flag_set = True
                great_flag = (r > great_flare_thresh)
                mode[i] = 2  # flare-high
            else:
                mode[i] = 0  # quiet
        else:
            if r > great_flare_thresh:
                great_flag = True
                mode[i] = 2
            elif r > flare_end_thresh:
                mode[i] = 1 if not great_flag else 2  # case B vs case C
            else:
                flag_set = False
                great_flag = False
                mode[i] = 0
    return mode


t = np.arange(0, 2400, 1.0)
rate = synthetic_flare_lightcurve(t)
mode = classify_modes(rate)

labels = {0: 'QUIET', 1: 'FLARE-MED', 2: 'FLARE-HIGH'}
colors = {0: 'tab:green', 1: 'tab:orange', 2: 'tab:red'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6),
                                sharex=True, gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(t, rate, color='black', lw=0.8)
ax1.axhline(250, ls='--', color='tab:orange', label='flare thr (250)')
ax1.axhline(150, ls='--', color='tab:green', label='flare-end thr (150)')
ax1.axhline(2000, ls='--', color='tab:red', label='great-flare thr (2000)')
ax1.set_ylabel('counts/s (synthetic HXS)')
ax1.set_yscale('log')
ax1.set_title('Simulated SOLAR-A DP flare-mode logic / SOLAR-A DP 플레어 모드 시뮬레이션')
ax1.legend(loc='upper left')

for m in [0, 1, 2]:
    sel = (mode == m)
    ax2.fill_between(t, m - 0.4, m + 0.4, where=sel,
                     color=colors[m], step='mid')
ax2.set_yticks([0, 1, 2]); ax2.set_yticklabels([labels[k] for k in [0, 1, 2]])
ax2.set_xlabel('time [s]')
ax2.set_xlim(0, t[-1])
plt.tight_layout()
plt.show()

n_high = int((mode == 2).sum())
n_med = int((mode == 1).sum())
print(f'Time in flare-high: {n_high} s; flare-medium: {n_med} s; '
      f'quiet: {(mode==0).sum()} s')

## Part 7: Solar Cycle 22 Context / 태양활동 22주기 맥락

SOLAR-A was designed to operate during **solar cycle 22 maximum (1989–1992)**. We illustrate the cycle's sunspot envelope and overlay SOLAR-A's launch and approximate operations. The synthetic curve uses a Hathaway-like analytical form parameterized by cycle 22 data.

SOLAR-A는 22주기 극대기에 맞춰 설계되었습니다. Hathaway 형태의 합성 곡선과 발사일을 비교하여 운용 시점의 활동 수준을 보입니다.

In [ ]:
def hathaway_cycle(t_yr: np.ndarray, t0: float, A: float = 200.0,
                   b: float = 60.0, c: float = 0.7) -> np.ndarray:
    """Compute Hathaway-style sunspot-number envelope of a cycle.

    Args:
        t_yr: Time in years.
        t0: Cycle start year.
        A: Amplitude in sunspot number.
        b: Rise time scale (months).
        c: Asymmetry parameter.

    Returns:
        Synthetic monthly sunspot number.
    """
    tm = (t_yr - t0) * 12.0  # months
    out = np.where(tm > 0,
                   A * (tm / b)**3 / (np.exp((tm / b)**2) - c),
                   0.0)
    return out


years = np.arange(1986, 2002, 1.0/12.0)
ssn22 = hathaway_cycle(years, t0=1986.7, A=160.0)
ssn23 = hathaway_cycle(years, t0=1996.7, A=120.0)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(years, ssn22, color='tab:orange', label='Cycle 22 (synthetic)')
ax.plot(years, ssn23, color='tab:blue', label='Cycle 23 (synthetic)')
ax.axvline(1991.66, color='red', ls='--', label='SOLAR-A launch (1991-08)')
ax.axvspan(1991.66, 2001.92, color='red', alpha=0.08,
           label='YOHKOH ops (1991-2001)')
ax.set_xlabel('Year'); ax.set_ylabel('Sunspot number (synthetic)')
ax.set_title('SOLAR-A / YOHKOH within Solar Cycles 22 & 23\n'
             'SOLAR-A/YOHKOH 운용 기간과 22-23주기')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ssn_at_launch = hathaway_cycle(np.array([1991.66]), t0=1986.7, A=160.0)[0]
print(f'Synthetic SSN at SOLAR-A launch: {ssn_at_launch:.0f} '
      f'(cycle 22 declining phase but still very active)')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Hard X-ray imaging | HXT — 64 bigrid collimators, 32 (u,v) samples, ~5″, 15–100 keV | RHESSI rotating-modulation imaging spectroscopy; STIX (Solar Orbiter) Fourier imaging |
| Soft X-ray imaging | SXT — modified Wolter-I + 1024×1024 CCD, ~2.5″, 3–60 Å | Hinode XRT; AIA (SDO) EUV, NuSTAR for hard X-ray |
| Bragg spectroscopy | BCS — 4 bent crystals, S XV / Ca XIX / Fe XXV / Fe XXVI | EIS (Hinode) UV spectroscopy; MinXSS CubeSat; FOXSI rocket |
| Broadband spectrometer | WBS — gas/NaI/BGO 2 keV–100 MeV | RHESSI Ge detectors; Fermi LAT for very high γ-rays |
| Onboard automation | DP triple-threshold flare-mode logic | SDO/AIA flare-watch; Hinode FOV management; STIX flare flag |
| Data overwrite priority | great > normal flare > quiet, BDR 10 MB | Hinode mass memory + DSN scheduling; PSP autonomy |
| Three-axis stabilization | 1″ pointing on Z-axis | SDO 0.05″; Hinode 0.1″; Solar Orbiter 6″ |

**Implementation files written / 구현 파일**: `41_ogawara_1991_implementation.ipynb` — 7 self-contained sections covering instrument coverage, orbit geometry, telemetry budget, BCS Doppler resolution, HXT Fourier-synthesis principle, automated flare-mode state machine, and solar-cycle-22 context.